In [ ]:
!pip install langchain transformers pypdf faiss-cpu sentence-transformers
!pip install langchain_community
!pip install langchain_huggingface

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Step 1: Load PDF as LangChain Documents
def load_pdf_as_documents(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

# Step 2: Split Documents into Chunks
def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_docs = text_splitter.split_documents(documents)
    return split_docs

# Step 3: Create FAISS Vector Store
def create_faiss_index_from_documents(documents):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(documents, embeddings)
    return vector_store

# Step 4: Set Up HuggingFace Question Generation
def setup_qg_pipeline():
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Step 5: RAG Pipeline Setup
def setup_rag_pipeline_with_documents(pdf_path):
    # Load PDF and split into chunks
    documents = load_pdf_as_documents(pdf_path)
    split_docs = split_documents_into_chunks(documents)

    # Create vector store
    vector_store = create_faiss_index_from_documents(split_docs)

    # Set up QA pipeline
    retriever = vector_store.as_retriever()
    qg_pipeline = setup_qg_pipeline()
    llm = HuggingFacePipeline(pipeline=qg_pipeline)
    qa_chain = RetrievalQA(llm=llm, retriever=retriever)

    return qa_chain



**Step 1** - Loading PDFs as LangChain Documents
Explanation:

This step uses PyPDFLoader to load a PDF and convert it into a list of Document objects.
Each Document contains text and optional metadata, such as page numbers.

In [ ]:
from langchain.document_loaders import PyPDFLoader

# Step 1: Load PDF as LangChain Documents
def load_pdf_as_documents(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

# Example: Load a PDF
pdf_path = "example.pdf"  # Replace with your PDF file
documents = load_pdf_as_documents(pdf_path)

# Display the first few documents
print("Number of documents loaded:", len(documents))
print("First document content:", documents[0].page_content[:500])  # Print first 500 characters


**Step 2** - Splitting Documents into Chunks
Explanation:

Large documents need to be divided into smaller chunks to improve retrieval performance.
RecursiveCharacterTextSplitter ensures that the text is split into manageable sizes while maintaining some overlap for context continuity.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 2: Split Documents into Chunks
def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_docs = text_splitter.split_documents(documents)
    return split_docs

# Example: Split the loaded documents
chunked_documents = split_documents_into_chunks(documents)
print("Number of chunks created:", len(chunked_documents))
print("First chunk content:", chunked_documents[0].page_content[:500])  # Print first 500 characters


**Step 3** - Creating a FAISS Vector Store
Explanation:

Text chunks are embedded into numerical vectors using sentence-transformers.
FAISS (Facebook AI Similarity Search) is used to index these vectors for fast retrieval during queries.

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# Step 3: Create FAISS Vector Store
def create_faiss_index_from_documents(documents):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(documents, embeddings)
    vector_store.save_local("faiss_store")
    return vector_store

# Example: Create a FAISS index
vector_store = create_faiss_index_from_documents(chunked_documents)
print("FAISS vector store created!")


In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Step 4: Set Up HuggingFace Question Generation
def setup_qg_pipeline():
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return pipeline("text2text-generation", model=model, tokenizer=tokenizer)

def setup_rag_pipeline_with_documents(pdf_path):
    # Load and process documents
    documents = load_pdf_as_documents(pdf_path)
    split_docs = split_documents_into_chunks(documents)
    vector_store = create_faiss_index_from_documents(split_docs)

    # Set up retriever
    retriever = vector_store.as_retriever()

    # Set up HuggingFacePipeline
    qg_pipeline = setup_qg_pipeline()
    llm = HuggingFacePipeline(pipeline=qg_pipeline)

    # Define a prompt template for QA
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template="Given the context: {context}, answer the question: {question}",
    )

    # Create RetrievalQA chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",  # Default chain type for combining documents
        retriever=retriever,
        return_source_documents=True,  # To get the source of the answer
        chain_type_kwargs={"prompt": prompt_template},
    )

    # Return the qa_chain
    return qa_chain  # Added this return statement

# Example: Set up the RAG pipeline
pdf_path = "example.pdf"  # Replace with your PDF file
qa_chain = setup_rag_pipeline_with_documents(pdf_path)

# Query the pipeline
query = "What is the document about?"  # Modify this query as needed
response = qa_chain.invoke(query)

print("\n--- Query Response ---")
print(response)
